# Incident Triage Evaluation Notebook\nColab-friendly quick evaluation for deterministic triage behavior.

In [ ]:
from pathlib import Path\nimport json\nfrom app.schemas.models import IncidentBundle\nfrom app.services.scenario_classifier import ScenarioClassifier\nfrom app.services.evidence_retrieval import EvidenceRetrieval\nfrom app.services.policy_engine import PolicyEngine

In [ ]:
sample_dir = Path('../data/sample_bundles')\nclassifier = ScenarioClassifier()\nretrieval = EvidenceRetrieval()\npolicy = PolicyEngine()\nrows = []\nfor path in sorted(sample_dir.glob('*.json')):\n    bundle = IncidentBundle.model_validate(json.loads(path.read_text()))\n    scenario = classifier.classify(bundle)\n    service = classifier.detect_likely_service(bundle)\n    evidence = retrieval.retrieve(bundle, scenario, service)\n    decision = policy.evaluate(bundle, scenario, service, evidence)\n    rows.append({\n        'sample': path.stem,\n        'incident_id': bundle.incident_id,\n        'scenario': scenario.value,\n        'severity': decision.severity.value,\n        'escalation': decision.escalation_target,\n        'score': decision.score,\n        'evidence_count': len(evidence),\n        'missing_context': decision.missing_context,\n    })\nrows